<a href="https://colab.research.google.com/github/msinan52/MIH/blob/main/4task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/MIH")
DATA_DIR = BASE_DIR / "etiketli"
LABEL_CSV = BASE_DIR / "label.csv"

print("BASE_DIR exists:", BASE_DIR.exists())
print("DATA_DIR exists:", DATA_DIR.exists())
print("LABEL_CSV exists:", LABEL_CSV.exists())

print("\nİlk birkaç dosya:")
for p in sorted(DATA_DIR.iterdir())[:10]:
    print(p.name)

BASE_DIR exists: True
DATA_DIR exists: True
LABEL_CSV exists: True

İlk birkaç dosya:
.DS_Store
1.JPG
1.json
10.JPG
10.json
100.JPG
100.json
101.jpeg
101.json
102.jpeg


In [ ]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# =========================================================
# AYARLAR
# =========================================================
BASE_DIR = Path("/content/drive/MyDrive/MIH")
DATA_DIR = BASE_DIR / "etiketli"
LABEL_CSV = BASE_DIR / "label.csv"

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 15
LR = 1e-4
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_SAVE_PATH = BASE_DIR / "mih_multitask_model.pth"

# =========================================================
# SEED
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

print("Device:", DEVICE)
print("BASE_DIR:", BASE_DIR)
print("DATA_DIR exists:", DATA_DIR.exists())
print("LABEL_CSV exists:", LABEL_CSV.exists())

# =========================================================
# CSV YÜKLE
# =========================================================
df = pd.read_csv(LABEL_CSV)
df.columns = [c.strip() for c in df.columns]

print("\nCSV sütunları:", list(df.columns))

required_cols = [
    "Image no",
    "INDEKS 1-R",
    "INDEKS 2-R",
    "INDEKS 2 GENİŞLİK-R",
    "INDEKS 3-R"
]

for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Eksik sütun: {c}")

# =========================================================
# GÖRSEL YOLUNU BUL
# =========================================================
def find_image_path(image_no):
    image_no = int(image_no)
    for ext in [".png", ".jpg", ".jpeg", ".bmp", ".webp"]:
        p = DATA_DIR / f"{image_no}{ext}"
        if p.exists():
            return str(p)
    return None

df["image_path"] = df["Image no"].apply(find_image_path)
df = df[df["image_path"].notnull()].copy().reset_index(drop=True)

print("\nToplam eşleşen örnek:", len(df))
print(df.head())

# =========================================================
# LABEL ENCODING
# =========================================================
target_cols = [
    "INDEKS 1-R",
    "INDEKS 2-R",
    "INDEKS 2 GENİŞLİK-R",
    "INDEKS 3-R"
]

label_maps = {}
inv_label_maps = {}
num_classes = {}

for col in target_cols:
    classes = sorted(df[col].dropna().unique().tolist())
    label_maps[col] = {v: i for i, v in enumerate(classes)}
    inv_label_maps[col] = {i: v for v, i in label_maps[col].items()}
    df[col + "_enc"] = df[col].map(label_maps[col]).astype(int)
    num_classes[col] = len(classes)

print("\nSınıf sayıları:", num_classes)
print("\nLabel maps:")
for k, v in label_maps.items():
    print(k, "->", v)

# =========================================================
# TRAIN / VAL SPLIT
# =========================================================
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    shuffle=True
)

print("\nTrain:", len(train_df), "Val:", len(val_df))

# =========================================================
# TRANSFORMS
# =========================================================
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# =========================================================
# DATASET
# =========================================================
class MIHDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")

        if self.transform:
            img = self.transform(img)

        y1 = int(row["INDEKS 1-R_enc"])
        y2 = int(row["INDEKS 2-R_enc"])
        y3 = int(row["INDEKS 2 GENİŞLİK-R_enc"])
        y4 = int(row["INDEKS 3-R_enc"])

        return img, (
            torch.tensor(y1, dtype=torch.long),
            torch.tensor(y2, dtype=torch.long),
            torch.tensor(y3, dtype=torch.long),
            torch.tensor(y4, dtype=torch.long),
        )

train_ds = MIHDataset(train_df, transform=train_tfms)
val_ds = MIHDataset(val_df, transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# =========================================================
# MODEL
# =========================================================
class MultiTaskResNet(nn.Module):
    def __init__(self, num_classes_dict):
        super().__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.head1 = nn.Linear(in_features, num_classes_dict["INDEKS 1-R"])
        self.head2 = nn.Linear(in_features, num_classes_dict["INDEKS 2-R"])
        self.head3 = nn.Linear(in_features, num_classes_dict["INDEKS 2 GENİŞLİK-R"])
        self.head4 = nn.Linear(in_features, num_classes_dict["INDEKS 3-R"])

    def forward(self, x):
        feat = self.backbone(x)
        out1 = self.head1(feat)
        out2 = self.head2(feat)
        out3 = self.head3(feat)
        out4 = self.head4(feat)
        return out1, out2, out3, out4

model = MultiTaskResNet(num_classes).to(DEVICE)

# =========================================================
# LOSS / OPTIMIZER
# =========================================================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# =========================================================
# TRAIN / VALID
# =========================================================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for images, targets in loader:
        images = images.to(device)
        y1, y2, y3, y4 = [t.to(device) for t in targets]

        optimizer.zero_grad()
        o1, o2, o3, o4 = model(images)

        loss1 = criterion(o1, y1)
        loss2 = criterion(o2, y2)
        loss3 = criterion(o3, y3)
        loss4 = criterion(o4, y4)

        loss = loss1 + loss2 + loss3 + loss4
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    all_true_1, all_pred_1 = [], []
    all_true_2, all_pred_2 = [], []
    all_true_3, all_pred_3 = [], []
    all_true_4, all_pred_4 = [], []

    for images, targets in loader:
        images = images.to(device)
        y1, y2, y3, y4 = [t.to(device) for t in targets]

        o1, o2, o3, o4 = model(images)

        loss1 = criterion(o1, y1)
        loss2 = criterion(o2, y2)
        loss3 = criterion(o3, y3)
        loss4 = criterion(o4, y4)
        loss = loss1 + loss2 + loss3 + loss4

        total_loss += loss.item() * images.size(0)

        p1 = torch.argmax(o1, dim=1)
        p2 = torch.argmax(o2, dim=1)
        p3 = torch.argmax(o3, dim=1)
        p4 = torch.argmax(o4, dim=1)

        all_true_1.extend(y1.cpu().numpy())
        all_pred_1.extend(p1.cpu().numpy())

        all_true_2.extend(y2.cpu().numpy())
        all_pred_2.extend(p2.cpu().numpy())

        all_true_3.extend(y3.cpu().numpy())
        all_pred_3.extend(p3.cpu().numpy())

        all_true_4.extend(y4.cpu().numpy())
        all_pred_4.extend(p4.cpu().numpy())

    acc1 = accuracy_score(all_true_1, all_pred_1)
    acc2 = accuracy_score(all_true_2, all_pred_2)
    acc3 = accuracy_score(all_true_3, all_pred_3)
    acc4 = accuracy_score(all_true_4, all_pred_4)
    mean_acc = (acc1 + acc2 + acc3 + acc4) / 4

    return total_loss / len(loader.dataset), acc1, acc2, acc3, acc4, mean_acc

# =========================================================
# EĞİTİM
# =========================================================
best_mean_acc = 0.0

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, acc1, acc2, acc3, acc4, mean_acc = validate(model, val_loader, criterion, DEVICE)

    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Acc INDEKS 1-R:          {acc1:.4f}")
    print(f"Acc INDEKS 2-R:          {acc2:.4f}")
    print(f"Acc INDEKS 2 GENİŞLİK-R: {acc3:.4f}")
    print(f"Acc INDEKS 3-R:          {acc4:.4f}")
    print(f"Mean Acc:                {mean_acc:.4f}")

    if mean_acc > best_mean_acc:
        best_mean_acc = mean_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "label_maps": label_maps,
            "inv_label_maps": inv_label_maps,
            "num_classes": num_classes,
            "img_size": IMG_SIZE
        }, MODEL_SAVE_PATH)
        print("En iyi model kaydedildi.")

print("\nEğitim bitti.")
print("Best mean acc:", best_mean_acc)
print("Model saved to:", MODEL_SAVE_PATH)

Device: cpu
BASE_DIR: /content/drive/MyDrive/MIH
DATA_DIR exists: True
LABEL_CSV exists: True

CSV sütunları: ['Image no', 'INDEKS 1-R', 'INDEKS 2-R', 'INDEKS 2 GENİŞLİK-R', 'INDEKS 3-R']

Toplam eşleşen örnek: 166
   Image no  INDEKS 1-R  INDEKS 2-R  INDEKS 2 GENİŞLİK-R  INDEKS 3-R  \
0         3           6           5                    3           1   
1        12           1           2                    1           0   
2        13           1           2                    1           0   
3        14           2           2                    2           0   
4        15           2           2                    2           0   

                                   image_path  
0  /content/drive/MyDrive/MIH/etiketli/3.jpeg  
1  /content/drive/MyDrive/MIH/etiketli/12.jpg  
2  /content/drive/MyDrive/MIH/etiketli/13.jpg  
3  /content/drive/MyDrive/MIH/etiketli/14.jpg  
4  /content/drive/MyDrive/MIH/etiketli/15.jpg  

Sınıf sayıları: {'INDEKS 1-R': 8, 'INDEKS 2-R': 5, 'INDEKS 2 GE

100%|██████████| 44.7M/44.7M [00:00<00:00, 148MB/s]



Epoch [1/15]
Train Loss: 4.9727
Val Loss:   4.3975
Acc INDEKS 1-R:          0.2941
Acc INDEKS 2-R:          0.6471
Acc INDEKS 2 GENİŞLİK-R: 0.5588
Acc INDEKS 3-R:          0.7059
Mean Acc:                0.5515
En iyi model kaydedildi.

Epoch [2/15]
Train Loss: 3.3275
Val Loss:   3.5802
Acc INDEKS 1-R:          0.5588
Acc INDEKS 2-R:          0.7059
Acc INDEKS 2 GENİŞLİK-R: 0.5000
Acc INDEKS 3-R:          0.7941
Mean Acc:                0.6397
En iyi model kaydedildi.

Epoch [3/15]
Train Loss: 2.7023
Val Loss:   3.2862
Acc INDEKS 1-R:          0.5294
Acc INDEKS 2-R:          0.7353
Acc INDEKS 2 GENİŞLİK-R: 0.5000
Acc INDEKS 3-R:          0.8824
Mean Acc:                0.6618
En iyi model kaydedildi.

Epoch [4/15]
Train Loss: 2.2272
Val Loss:   3.2334
Acc INDEKS 1-R:          0.4706
Acc INDEKS 2-R:          0.7647
Acc INDEKS 2 GENİŞLİK-R: 0.5588
Acc INDEKS 3-R:          0.8235
Mean Acc:                0.6544

Epoch [5/15]
Train Loss: 1.6619
Val Loss:   3.1771
Acc INDEKS 1-R:          

In [ ]:
import torch
from PIL import Image
from torchvision import transforms

checkpoint = torch.load("/content/drive/MyDrive/MIH/mih_multitask_model.pth", map_location=DEVICE)

model = MultiTaskResNet(checkpoint["num_classes"]).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

inv_label_maps = checkpoint["inv_label_maps"]
img_size = checkpoint["img_size"]

infer_tfms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def predict_image(img_path):
    img = Image.open(img_path).convert("RGB")
    x = infer_tfms(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        o1, o2, o3, o4 = model(x)

    p1 = torch.argmax(o1, dim=1).item()
    p2 = torch.argmax(o2, dim=1).item()
    p3 = torch.argmax(o3, dim=1).item()
    p4 = torch.argmax(o4, dim=1).item()

    return {
        "INDEKS 1-R": inv_label_maps["INDEKS 1-R"][p1],
        "INDEKS 2-R": inv_label_maps["INDEKS 2-R"][p2],
        "INDEKS 2 GENİŞLİK-R": inv_label_maps["INDEKS 2 GENİŞLİK-R"][p3],
        "INDEKS 3-R": inv_label_maps["INDEKS 3-R"][p4],
    }

sample_path = "/content/drive/MyDrive/MIH/etiketli/258.png"
print(predict_image(sample_path))

{'INDEKS 1-R': 6, 'INDEKS 2-R': 5, 'INDEKS 2 GENİŞLİK-R': 3, 'INDEKS 3-R': 1}


#transfer learning

In [ ]:
class MultiTaskModel(nn.Module):
    def __init__(self, num_classes_dict, model_name="efficientnet"):
        super().__init__()

        if model_name == "resnet":
            backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
            in_features = backbone.fc.in_features
            backbone.fc = nn.Identity()

        elif model_name == "vgg":
            backbone = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
            in_features = backbone.classifier[-1].in_features
            backbone.classifier = nn.Sequential(*list(backbone.classifier.children())[:-1])

        elif model_name == "efficientnet":
            backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
            in_features = backbone.classifier[1].in_features
            backbone.classifier = nn.Identity()

        elif model_name == "densenet":
            backbone = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
            in_features = backbone.classifier.in_features
            backbone.classifier = nn.Identity()

        else:
            raise ValueError("Model bulunamadı")

        self.backbone = backbone

        # 4 task
        self.head1 = nn.Linear(in_features, num_classes_dict["INDEKS 1-R"])
        self.head2 = nn.Linear(in_features, num_classes_dict["INDEKS 2-R"])
        self.head3 = nn.Linear(in_features, num_classes_dict["INDEKS 2 GENİŞLİK-R"])
        self.head4 = nn.Linear(in_features, num_classes_dict["INDEKS 3-R"])

    def forward(self, x):
        feat = self.backbone(x)
        return (
            self.head1(feat),
            self.head2(feat),
            self.head3(feat),
            self.head4(feat),
        )

In [ ]:
results = {}

for MODEL_NAME in ["resnet", "vgg", "efficientnet", "densenet"]:
    print("\nMODEL:", MODEL_NAME)

    model = MultiTaskModel(num_classes, model_name=MODEL_NAME).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_acc = 0

    for epoch in range(5):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss, acc1, acc2, acc3, acc4, mean_acc = validate(model, val_loader, criterion, DEVICE)

        print(f"{MODEL_NAME} Epoch {epoch+1}: {mean_acc:.4f}")

        best_acc = max(best_acc, mean_acc)

    results[MODEL_NAME] = best_acc

print("\nSONUÇLAR:")
print(results)


MODEL: resnet
resnet Epoch 1: 0.5294
resnet Epoch 2: 0.5956
resnet Epoch 3: 0.6618
resnet Epoch 4: 0.6618
resnet Epoch 5: 0.6765

MODEL: vgg
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:05<00:00, 101MB/s] 


vgg Epoch 1: 0.4338
vgg Epoch 2: 0.3529
vgg Epoch 3: 0.6029
vgg Epoch 4: 0.5882
vgg Epoch 5: 0.5956

MODEL: efficientnet
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 139MB/s]


efficientnet Epoch 1: 0.4779
efficientnet Epoch 2: 0.4853
efficientnet Epoch 3: 0.5588
efficientnet Epoch 4: 0.5368
efficientnet Epoch 5: 0.5588

MODEL: densenet
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 134MB/s]


densenet Epoch 1: 0.5147
densenet Epoch 2: 0.5221
densenet Epoch 3: 0.5956
densenet Epoch 4: 0.5956
densenet Epoch 5: 0.5809

SONUÇLAR:
{'resnet': 0.6764705882352942, 'vgg': 0.6029411764705882, 'efficientnet': 0.5588235294117647, 'densenet': 0.5955882352941176}


#grafikler

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

In [ ]:
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    correct1 = correct2 = correct3 = correct4 = 0
    total = 0

    all_labels_task1, all_preds_task1, all_probs_task1 = [], [], []
    all_labels_task2, all_preds_task2, all_probs_task2 = [], [], []
    all_labels_task3, all_preds_task3, all_probs_task3 = [], [], []
    all_labels_task4, all_preds_task4, all_probs_task4 = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            y1 = labels[:, 0]
            y2 = labels[:, 1]
            y3 = labels[:, 2]
            y4 = labels[:, 3]

            out1, out2, out3, out4 = model(images)

            loss1 = criterion(out1, y1)
            loss2 = criterion(out2, y2)
            loss3 = criterion(out3, y3)
            loss4 = criterion(out4, y4)
            loss = loss1 + loss2 + loss3 + loss4

            total_loss += loss.item()

            pred1 = torch.argmax(out1, dim=1)
            pred2 = torch.argmax(out2, dim=1)
            pred3 = torch.argmax(out3, dim=1)
            pred4 = torch.argmax(out4, dim=1)

            correct1 += (pred1 == y1).sum().item()
            correct2 += (pred2 == y2).sum().item()
            correct3 += (pred3 == y3).sum().item()
            correct4 += (pred4 == y4).sum().item()
            total += y1.size(0)

            prob1 = torch.softmax(out1, dim=1)
            prob2 = torch.softmax(out2, dim=1)
            prob3 = torch.softmax(out3, dim=1)
            prob4 = torch.softmax(out4, dim=1)

            all_labels_task1.extend(y1.cpu().numpy())
            all_preds_task1.extend(pred1.cpu().numpy())
            all_probs_task1.extend(prob1.cpu().numpy())

            all_labels_task2.extend(y2.cpu().numpy())
            all_preds_task2.extend(pred2.cpu().numpy())
            all_probs_task2.extend(prob2.cpu().numpy())

            all_labels_task3.extend(y3.cpu().numpy())
            all_preds_task3.extend(pred3.cpu().numpy())
            all_probs_task3.extend(prob3.cpu().numpy())

            all_labels_task4.extend(y4.cpu().numpy())
            all_preds_task4.extend(pred4.cpu().numpy())
            all_probs_task4.extend(prob4.cpu().numpy())

    acc1 = correct1 / total
    acc2 = correct2 / total
    acc3 = correct3 / total
    acc4 = correct4 / total
    mean_acc = (acc1 + acc2 + acc3 + acc4) / 4
    avg_loss = total_loss / len(loader)

    details = {
        "task1": {
            "labels": np.array(all_labels_task1),
            "preds": np.array(all_preds_task1),
            "probs": np.array(all_probs_task1)
        },
        "task2": {
            "labels": np.array(all_labels_task2),
            "preds": np.array(all_preds_task2),
            "probs": np.array(all_probs_task2)
        },
        "task3": {
            "labels": np.array(all_labels_task3),
            "preds": np.array(all_preds_task3),
            "probs": np.array(all_probs_task3)
        },
        "task4": {
            "labels": np.array(all_labels_task4),
            "preds": np.array(all_preds_task4),
            "probs": np.array(all_probs_task4)
        }
    }

    return avg_loss, acc1, acc2, acc3, acc4, mean_acc, details

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize


# =========================
# LABEL AYRIŞTIRMA YARDIMCISI
# =========================
def unpack_labels(labels, device):
    """
    labels şu formatlardan biri olabilir:
    1) Tensor: [B, 4]
    2) List of Tensor: [y1_batch, y2_batch, y3_batch, y4_batch]
    3) List/Numpy: [[y1,y2,y3,y4], ...]
    """

    if torch.is_tensor(labels):
        labels = labels.long().to(device)
        if labels.ndim == 1:
            raise ValueError("labels tensor shape [B,4] olmalı, ama [B] geldi.")
        if labels.shape[1] < 4:
            raise ValueError(f"labels ikinci boyutu en az 4 olmalı, gelen shape: {labels.shape}")
        y1 = labels[:, 0]
        y2 = labels[:, 1]
        y3 = labels[:, 2]
        y4 = labels[:, 3]
        return y1, y2, y3, y4

    if isinstance(labels, (list, tuple)):
        # Durum: [y1_batch, y2_batch, y3_batch, y4_batch]
        if len(labels) == 4 and all(torch.is_tensor(x) for x in labels):
            y1 = labels[0].long().to(device)
            y2 = labels[1].long().to(device)
            y3 = labels[2].long().to(device)
            y4 = labels[3].long().to(device)
            return y1, y2, y3, y4

        # Durum: [[y1,y2,y3,y4], [y1,y2,y3,y4], ...]
        labels = torch.tensor(labels, dtype=torch.long, device=device)
        if labels.ndim == 1:
            raise ValueError("labels list tensor'a çevrildi ama shape [B] oldu. [B,4] bekleniyor.")
        if labels.shape[1] < 4:
            raise ValueError(f"labels ikinci boyutu en az 4 olmalı, gelen shape: {labels.shape}")
        y1 = labels[:, 0]
        y2 = labels[:, 1]
        y3 = labels[:, 2]
        y4 = labels[:, 3]
        return y1, y2, y3, y4

    raise TypeError(f"Desteklenmeyen labels tipi: {type(labels)}")


# =========================
# TRAIN
# =========================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for images, labels in loader:
        images = images.to(device)
        y1, y2, y3, y4 = unpack_labels(labels, device)

        optimizer.zero_grad()

        out1, out2, out3, out4 = model(images)

        loss1 = criterion(out1, y1)
        loss2 = criterion(out2, y2)
        loss3 = criterion(out3, y3)
        loss4 = criterion(out4, y4)

        loss = loss1 + loss2 + loss3 + loss4
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


# =========================
# VALIDATE
# =========================
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    correct1 = correct2 = correct3 = correct4 = 0
    total = 0

    all_labels_task1, all_preds_task1, all_probs_task1 = [], [], []
    all_labels_task2, all_preds_task2, all_probs_task2 = [], [], []
    all_labels_task3, all_preds_task3, all_probs_task3 = [], [], []
    all_labels_task4, all_preds_task4, all_probs_task4 = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            y1, y2, y3, y4 = unpack_labels(labels, device)

            out1, out2, out3, out4 = model(images)

            loss1 = criterion(out1, y1)
            loss2 = criterion(out2, y2)
            loss3 = criterion(out3, y3)
            loss4 = criterion(out4, y4)
            loss = loss1 + loss2 + loss3 + loss4

            total_loss += loss.item()

            pred1 = torch.argmax(out1, dim=1)
            pred2 = torch.argmax(out2, dim=1)
            pred3 = torch.argmax(out3, dim=1)
            pred4 = torch.argmax(out4, dim=1)

            correct1 += (pred1 == y1).sum().item()
            correct2 += (pred2 == y2).sum().item()
            correct3 += (pred3 == y3).sum().item()
            correct4 += (pred4 == y4).sum().item()
            total += y1.size(0)

            prob1 = torch.softmax(out1, dim=1)
            prob2 = torch.softmax(out2, dim=1)
            prob3 = torch.softmax(out3, dim=1)
            prob4 = torch.softmax(out4, dim=1)

            all_labels_task1.extend(y1.cpu().numpy())
            all_preds_task1.extend(pred1.cpu().numpy())
            all_probs_task1.extend(prob1.cpu().numpy())

            all_labels_task2.extend(y2.cpu().numpy())
            all_preds_task2.extend(pred2.cpu().numpy())
            all_probs_task2.extend(prob2.cpu().numpy())

            all_labels_task3.extend(y3.cpu().numpy())
            all_preds_task3.extend(pred3.cpu().numpy())
            all_probs_task3.extend(prob3.cpu().numpy())

            all_labels_task4.extend(y4.cpu().numpy())
            all_preds_task4.extend(pred4.cpu().numpy())
            all_probs_task4.extend(prob4.cpu().numpy())

    acc1 = correct1 / total
    acc2 = correct2 / total
    acc3 = correct3 / total
    acc4 = correct4 / total
    mean_acc = (acc1 + acc2 + acc3 + acc4) / 4
    avg_loss = total_loss / len(loader)

    details = {
        "task1": {
            "labels": np.array(all_labels_task1),
            "preds": np.array(all_preds_task1),
            "probs": np.array(all_probs_task1)
        },
        "task2": {
            "labels": np.array(all_labels_task2),
            "preds": np.array(all_preds_task2),
            "probs": np.array(all_probs_task2)
        },
        "task3": {
            "labels": np.array(all_labels_task3),
            "preds": np.array(all_preds_task3),
            "probs": np.array(all_probs_task3)
        },
        "task4": {
            "labels": np.array(all_labels_task4),
            "preds": np.array(all_preds_task4),
            "probs": np.array(all_probs_task4)
        }
    }

    return avg_loss, acc1, acc2, acc3, acc4, mean_acc, details


# =========================
# GRAFİK: HISTORY
# =========================
def plot_history(history):
    for model_name in history:
        epochs = range(1, len(history[model_name]["train_loss"]) + 1)

        plt.figure(figsize=(14, 5))

        # Loss
        plt.subplot(1, 2, 1)
        plt.plot(epochs, history[model_name]["train_loss"], marker='o', label="Train Loss")
        plt.plot(epochs, history[model_name]["val_loss"], marker='o', label="Val Loss")
        plt.title(f"{model_name} Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid(True)

        # Accuracy
        plt.subplot(1, 2, 2)
        plt.plot(epochs, history[model_name]["acc1"], marker='o', label="Task1 Acc")
        plt.plot(epochs, history[model_name]["acc2"], marker='o', label="Task2 Acc")
        plt.plot(epochs, history[model_name]["acc3"], marker='o', label="Task3 Acc")
        plt.plot(epochs, history[model_name]["acc4"], marker='o', label="Task4 Acc")
        plt.plot(epochs, history[model_name]["mean_acc"], marker='o', label="Mean Acc")
        plt.title(f"{model_name} Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        plt.show()


# =========================
# CONFUSION MATRIX
# =========================
def plot_confusion_matrices(best_details, best_model_name, class_names=None):
    for task_name in ["task1", "task2", "task3", "task4"]:
        y_true = best_details[task_name]["labels"]
        y_pred = best_details[task_name]["preds"]

        cm = confusion_matrix(y_true, y_pred)

        plt.figure(figsize=(6, 6))
        if class_names is not None:
            disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
        else:
            disp = ConfusionMatrixDisplay(confusion_matrix=cm)

        disp.plot(values_format='d')
        plt.title(f"{best_model_name} - {task_name} Confusion Matrix")
        plt.show()


# =========================
# ROC
# =========================
def plot_roc_curves(best_details, best_model_name):
    for task_name in ["task1", "task2", "task3", "task4"]:
        y_true = best_details[task_name]["labels"]
        y_score = best_details[task_name]["probs"]

        if len(y_score.shape) != 2:
            print(f"{task_name} için ROC çizilemedi: probs shape hatalı -> {y_score.shape}")
            continue

        n_classes = y_score.shape[1]

        # Eğer tek sınıf geldiyse ROC anlamsız olur
        unique_classes = np.unique(y_true)
        if len(unique_classes) < 2:
            print(f"{task_name} için ROC çizilemedi: tek sınıf var -> {unique_classes}")
            continue

        y_true_bin = label_binarize(y_true, classes=np.arange(n_classes))

        plt.figure(figsize=(7, 6))
        plotted = False

        for i in range(n_classes):
            # Bu sınıfa ait pozitif örnek yoksa ROC hesaplanmaz
            if y_true_bin[:, i].sum() == 0:
                continue

            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f"Class {i} (AUC = {roc_auc:.3f})")
            plotted = True

        if not plotted:
            print(f"{task_name} için ROC çizilemedi: uygun sınıf dağılımı yok.")
            plt.close()
            continue

        plt.plot([0, 1], [0, 1], linestyle='--')
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"{best_model_name} - {task_name} ROC Curve")
        plt.legend()
        plt.grid(True)
        plt.show()


# =========================
# ANA EĞİTİM DÖNGÜSÜ
# =========================
results = {}
history = {}

model_names = ["resnet", "vgg", "efficientnet", "densenet"]

for MODEL_NAME in model_names:
    print("\n" + "=" * 50)
    print("MODEL:", MODEL_NAME)
    print("=" * 50)

    model = MultiTaskModel(num_classes, model_name=MODEL_NAME).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_acc = 0.0
    best_details = None
    best_state_dict = None

    history[MODEL_NAME] = {
        "train_loss": [],
        "val_loss": [],
        "acc1": [],
        "acc2": [],
        "acc3": [],
        "acc4": [],
        "mean_acc": []
    }

    for epoch in range(5):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss, acc1, acc2, acc3, acc4, mean_acc, details = validate(
            model, val_loader, criterion, DEVICE
        )

        history[MODEL_NAME]["train_loss"].append(train_loss)
        history[MODEL_NAME]["val_loss"].append(val_loss)
        history[MODEL_NAME]["acc1"].append(acc1)
        history[MODEL_NAME]["acc2"].append(acc2)
        history[MODEL_NAME]["acc3"].append(acc3)
        history[MODEL_NAME]["acc4"].append(acc4)
        history[MODEL_NAME]["mean_acc"].append(mean_acc)

        print(
            f"{MODEL_NAME} Epoch {epoch+1} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Acc1: {acc1:.4f} | Acc2: {acc2:.4f} | "
            f"Acc3: {acc3:.4f} | Acc4: {acc4:.4f} | "
            f"Mean Acc: {mean_acc:.4f}"
        )

        if mean_acc > best_acc:
            best_acc = mean_acc
            best_details = details
            best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    results[MODEL_NAME] = {
        "best_acc": best_acc,
        "best_details": best_details,
        "best_state_dict": best_state_dict
    }

print("\nSONUÇLAR:")
for model_name, info in results.items():
    print(f"{model_name}: {info['best_acc']:.4f}")


# =========================
# HISTORY GRAFİKLERİ
# =========================
plot_history(history)


# =========================
# EN İYİ MODELİ BUL
# =========================
best_model_name = max(results, key=lambda x: results[x]["best_acc"])
best_details = results[best_model_name]["best_details"]

print("\nEn iyi model:", best_model_name)
print("En iyi mean accuracy:", results[best_model_name]["best_acc"])


# =========================
# CONFUSION MATRIX
# =========================
plot_confusion_matrices(best_details, best_model_name, class_names=None)


# =========================
# ROC
# =========================
plot_roc_curves(best_details, best_model_name)

#segmentasyon

In [ ]:
import json
import numpy as np
from PIL import Image, ImageDraw
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/MIH/etiketli")
MASK_DIR = Path("/content/drive/MyDrive/MIH/masks")
MASK_DIR.mkdir(exist_ok=True)

def json_to_mask(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)

    h = data.get("imageHeight")
    w = data.get("imageWidth")

    mask = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(mask)

    for shape in data.get("shapes", []):
        points = shape["points"]
        polygon = [tuple(p) for p in points]

        if len(polygon) >= 3:
            draw.polygon(polygon, outline=1, fill=1)

    return mask

for jf in DATA_DIR.glob("*.json"):
    mask = json_to_mask(jf)
    mask.save(MASK_DIR / f"{jf.stem}.png")

print("Mask üretildi")

#unet model

In [ ]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.ReLU(),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.ReLU()
            )

        self.enc1 = block(3, 64)
        self.enc2 = block(64, 128)
        self.enc3 = block(128, 256)

        self.pool = nn.MaxPool2d(2)

        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec1 = block(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = block(128, 64)

        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        d1 = self.up1(e3)
        d1 = self.dec1(torch.cat([d1, e2], dim=1))

        d2 = self.up2(d1)
        d2 = self.dec2(torch.cat([d2, e1], dim=1))

        return torch.sigmoid(self.final(d2))

In [ ]:
def dice_loss(pred, target, smooth=1):
    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()
    return 1 - (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

bce = nn.BCELoss()

def loss_fn(pred, target):
    return bce(pred, target) + dice_loss(pred, target)

In [ ]:
model = UNet().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(10):
    model.train()
    total_loss = 0

    for imgs, masks in train_loader:
        imgs = imgs.to(DEVICE)
        masks = masks.to(DEVICE)

        preds = model(imgs)

        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: {total_loss:.4f}")

#XAI

In [ ]:
import torch
import torch.nn.functional as F
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        self.forward_hook = target_layer.register_forward_hook(self.save_activation)
        self.backward_hook = target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output.detach()

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, x, task_idx=0, class_idx=None):
        self.model.eval()

        outputs = self.model(x)
        target_output = outputs[task_idx]

        if class_idx is None:
            class_idx = torch.argmax(target_output, dim=1).item()

        score = target_output[:, class_idx]
        self.model.zero_grad()
        score.backward(retain_graph=True)

        grads = self.gradients
        acts = self.activations

        weights = grads.mean(dim=(2, 3), keepdim=True)
        cam = (weights * acts).sum(dim=1, keepdim=True)
        cam = F.relu(cam)

        cam = F.interpolate(cam, size=x.shape[2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam, class_idx

    def close(self):
        self.forward_hook.remove()
        self.backward_hook.remove()

In [ ]:
# ResNet için
target_layer = model.backbone.layer4[-1]

# EfficientNet-B0 için genelde
# target_layer = model.backbone.features[-1]

# VGG için
# target_layer = model.backbone.features[-1]

In [ ]:
def show_gradcam(img_path, model, target_layer, task_idx=0, transform=None, alpha=0.4):
    if transform is None:
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

    raw_img = Image.open(img_path).convert("RGB")
    img_resized = raw_img.resize((224, 224))
    x = transform(raw_img).unsqueeze(0).to(next(model.parameters()).device)

    cam_engine = GradCAM(model, target_layer)
    cam, pred_class = cam_engine.generate(x, task_idx=task_idx)
    cam_engine.close()

    img_np = np.array(img_resized).astype(np.float32) / 255.0
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    overlay = heatmap * alpha + img_np * (1 - alpha)
    overlay = np.clip(overlay, 0, 1)

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(img_np)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(cam, cmap="jet")
    plt.title(f"Grad-CAM Task {task_idx+1}")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(overlay)
    plt.title(f"Overlay | Pred class: {pred_class}")
    plt.axis("off")

    plt.show()

In [ ]:
sample_img = "/content/drive/MyDrive/MIH/etiketli/258.png"
show_gradcam(sample_img, model, target_layer, task_idx=0)

In [ ]:
import torch

def dice_score(pred, target, smooth=1e-6):
    pred = (pred > 0.5).float()
    target = (target > 0.5).float()

    intersection = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))

    dice = (2.0 * intersection + smooth) / (union + smooth)
    return dice.mean().item()

def iou_score(pred, target, smooth=1e-6):
    pred = (pred > 0.5).float()
    target = (target > 0.5).float()

    intersection = (pred * target).sum(dim=(1,2,3))
    total = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    union = total - intersection

    iou = (intersection + smooth) / (union + smooth)
    return iou.mean().item()

#attention unet

In [ ]:
import torch
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class AttentionBlock(nn.Module):
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(g_ch, inter_ch, kernel_size=1),
            nn.BatchNorm2d(inter_ch)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(x_ch, inter_ch, kernel_size=1),
            nn.BatchNorm2d(inter_ch)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(inter_ch, 1, kernel_size=1),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class AttentionUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()

        self.pool = nn.MaxPool2d(2)

        self.enc1 = ConvBlock(in_ch, 64)
        self.enc2 = ConvBlock(64, 128)
        self.enc3 = ConvBlock(128, 256)
        self.enc4 = ConvBlock(256, 512)

        self.bottleneck = ConvBlock(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.att4 = AttentionBlock(512, 512, 256)
        self.dec4 = ConvBlock(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.att3 = AttentionBlock(256, 256, 128)
        self.dec3 = ConvBlock(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.att2 = AttentionBlock(128, 128, 64)
        self.dec2 = ConvBlock(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.att1 = AttentionBlock(64, 64, 32)
        self.dec1 = ConvBlock(128, 64)

        self.final = nn.Conv2d(64, out_ch, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        e4_att = self.att4(d4, e4)
        d4 = self.dec4(torch.cat([d4, e4_att], dim=1))

        d3 = self.up3(d4)
        e3_att = self.att3(d3, e3)
        d3 = self.dec3(torch.cat([d3, e3_att], dim=1))

        d2 = self.up2(d3)
        e2_att = self.att2(d2, e2)
        d2 = self.dec2(torch.cat([d2, e2_att], dim=1))

        d1 = self.up1(d2)
        e1_att = self.att1(d1, e1)
        d1 = self.dec1(torch.cat([d1, e1_att], dim=1))

        return torch.sigmoid(self.final(d1))

In [ ]:
seg_model = AttentionUNet().to(DEVICE)

In [ ]:
!pip install -q segmentation-models-pytorch

In [ ]:
import segmentation_models_pytorch as smp

seg_model = smp.UnetPlusPlus(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation="sigmoid"
).to(DEVICE)

In [ ]:
bce = nn.BCELoss()

def dice_loss(pred, target, smooth=1e-6):
    pred = pred.contiguous().view(pred.size(0), -1)
    target = target.contiguous().view(target.size(0), -1)

    intersection = (pred * target).sum(dim=1)
    dice = (2. * intersection + smooth) / (pred.sum(dim=1) + target.sum(dim=1) + smooth)
    return 1 - dice.mean()

def seg_loss_fn(pred, target):
    return bce(pred, target) + dice_loss(pred, target)

#ensemble

In [ ]:
import numpy as np

def mask_area_ratio(pred_mask):
    pred_mask = (pred_mask > 0.5).astype(np.uint8)
    return pred_mask.sum() / pred_mask.size

In [ ]:
def ensemble_predict(img_path, cls_model, seg_model, cls_transform, seg_transform, device):
    cls_model.eval()
    seg_model.eval()

    img = Image.open(img_path).convert("RGB")

    # classification
    x_cls = cls_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        cls_outputs = cls_model(x_cls)
        cls_preds = [torch.argmax(o, dim=1).item() for o in cls_outputs]

    # segmentation
    x_seg = seg_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        seg_pred = seg_model(x_seg)

    seg_mask = seg_pred.squeeze().cpu().numpy()
    area = mask_area_ratio(seg_mask)

    # basit kural tabanı örneği
    refined_preds = cls_preds.copy()

    # örnek: lezyon alanı çok büyükse belirli task'lerde sınıfı yukarı çek
    if area > 0.20:
        pass  # burada kendi klinik mantığını ya da öğrenilmiş kuralını koyarsın

    return {
        "cls_preds": cls_preds,
        "lesion_area_ratio": area,
        "refined_preds": refined_preds,
        "seg_mask": seg_mask
    }